In [ ]:
# STEP 1: basic setup – install libs (if needed), imports, GPU check

# If you already installed these earlier, this will just confirm/upgrade them.
!pip install -q numpy matplotlib rasterio 
!pip install tqdm

import sys, platform
print("Python:", sys.version.split()[0])
print("OS:", platform.platform())

#import numpy as np
#import matplotlib.pyplot as plt
#import rasterio
#from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import random, numpy as np, torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
# ============================================================
# EVAL-ONLY PIPELINE (NO TRAINING)
# - recompute train stats (RGBNIR mean/std) from RAW TRAIN ONLY
# - compute DEM_CENTER_MEDIAN from TRAIN ONLY
# - load BEST checkpoint
# - print VAL + TEST metrics (global pixel-wise, masked)
# - print BW inference metrics (4 npz)
# ============================================================

# ----------------------------
# STEP 0: Setup
# ----------------------------
!pip install -q numpy matplotlib rasterio
!pip install -q tqdm

import sys, platform, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torch.amp import autocast

print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")
print("Using device:", device, "| AMP:", use_amp)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ----------------------------
# STEP 1: Paths (SAME AS YOURS)
# ----------------------------
PATCH_ROOT = Path(r"D:\Sunita_Thesis\Datasets\Data_Patches_2km_arrays")
META_PATH  = PATCH_ROOT / "patch_metadata_with_order.csv"

RUN_DIR = PATCH_ROOT / "runs_unet_dem_normalized150_BW"
BEST_PATH = RUN_DIR / "checkpoint_best.pt"

print("PATCH_ROOT:", PATCH_ROOT)
print("META_PATH :", META_PATH)
print("BEST_PATH :", BEST_PATH)
assert META_PATH.exists(), f"Missing metadata: {META_PATH}"
assert BEST_PATH.exists(), f"Missing checkpoint: {BEST_PATH}"

# Inference dir (BW)
INF_DIR = Path(r"D:\Sunita_Thesis\Datasets\Inference data\DATASET-BW\DATASET-BW\_patches_npz")
INF_FILES = sorted(INF_DIR.glob("BW_AOI_*.npz"))
print("BW npz files:", [p.name for p in INF_FILES])
assert len(INF_FILES) > 0, f"No BW npz files found in {INF_DIR}"


# ----------------------------
# STEP 2: Split metadata (SAME CHECKERBOARD)
# ----------------------------
meta = pd.read_csv(META_PATH, sep=";")

rows_sorted = sorted(meta["row"].unique(), reverse=True)
cols_sorted = sorted(meta["col"].unique())

row_to_grid = {r: i for i, r in enumerate(rows_sorted)}
col_to_grid = {c: j for j, c in enumerate(cols_sorted)}

meta["grid_row"] = meta["row"].map(row_to_grid)
meta["grid_col"] = meta["col"].map(col_to_grid)

BLOCK = 4
block_row = meta["grid_row"] // BLOCK
block_col = meta["grid_col"] // BLOCK
gid = (block_row + block_col) % 7

def assign_split_from_gid(g):
    if g in {0, 1, 2, 3, 4}:
        return "train"
    elif g == 5:
        return "val"
    else:
        return "test"

meta["split"] = gid.apply(assign_split_from_gid)
meta_split = meta.copy()

print("Split counts:\n", meta_split["split"].value_counts())


# ----------------------------
# STEP 3: Constants (SAME)
# ----------------------------
# Each .npy: (6,200,200) = [DEM, R, G, B, NIR, NOISE]
INPUT_CHANNELS = [1, 2, 3, 4, 0]     # [R,G,B,NIR,DEM]
TARGET_CHANNEL = 5                  # NOISE
TARGET_NODATA  = 0.0                # your setting
MASK_ZERO_TARGET = True

DEM_RAW_CHANNEL = 0
DEM_NODATA = -9999.0
DEM_SCALE_FIXED = 50.0              # your fixed scale used in training


# ----------------------------
# STEP 4: Mask + Dataset (SAME LOGIC)
# ----------------------------
def build_target_mask(y_np: np.ndarray, nodata_value=None) -> np.ndarray:
    mask = np.isfinite(y_np)
    if nodata_value is not None:
        mask = mask & (y_np != nodata_value)
    mask = mask & (y_np > -1e6)
    if MASK_ZERO_TARGET:
        mask = mask & (y_np != 0.0)
    return mask.astype("float32")


class NpyPatchDataset(Dataset):
    """
    Each .npy: (6,200,200) = [DEM, R, G, B, NIR, NOISE]
    Returns:
      x: (5,200,200)  [R,G,B,NIR,DEM]  (RGBNIR normalized, DEM scaled)
      y: (1,200,200)  invalid filled 0
      mask: (1,200,200) valid=1, invalid=0
    """
    def __init__(self, meta_df, split: str, root: Path,
                 target_nodata=None, x_mean=None, x_std=None,
                 augment=False, dem_center=None, dem_scale=None):
        self.root = root
        self.meta = meta_df[meta_df["split"] == split].reset_index(drop=True)
        self.split = split
        self.target_nodata = target_nodata
        self.x_mean = x_mean
        self.x_std = x_std
        self.augment = augment
        self.dem_center = dem_center
        self.dem_scale  = dem_scale

        print(f"{split}: {len(self.meta)} patches | augment={augment}")

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        arr = np.load(self.root / row["filename"])  # (6,200,200)

        x = arr[INPUT_CHANNELS].astype("float32")  # (5,200,200)
        y = arr[TARGET_CHANNEL:TARGET_CHANNEL+1].astype("float32")  # (1,200,200)

        mask = build_target_mask(y, nodata_value=self.target_nodata)

        # DEM fill (median per patch) — SAME AS YOUR CODE
        dem_in_x_index = INPUT_CHANNELS.index(DEM_RAW_CHANNEL)
        dem = x[dem_in_x_index]
        dem_valid = np.isfinite(dem) & (dem != DEM_NODATA)
        dem_fill = np.median(dem[dem_valid]) if dem_valid.any() else 0.0
        dem_filled = dem.copy()
        dem_filled[~dem_valid] = dem_fill
        x[dem_in_x_index] = dem_filled

        # DEM scaling — SAME AS YOUR CODE
        if (self.dem_center is not None) and (self.dem_scale is not None):
            x[dem_in_x_index] = (x[dem_in_x_index] - self.dem_center) / (self.dem_scale + 1e-6)

        # Fill invalid target with 0 — SAME
        y_filled = y.copy()
        y_filled[mask == 0] = 0.0

        # Tensor
        x = torch.from_numpy(x)
        y_filled = torch.from_numpy(y_filled)
        mask = torch.from_numpy(mask)

        # Normalize RGBNIR only (channels 0..3), DEM left as already scaled — SAME
        if self.x_mean is not None and self.x_std is not None:
            mean = self.x_mean.view(4,1,1)
            std  = self.x_std.view(4,1,1)
            x[:4] = (x[:4] - mean) / (std + 1e-6)

        info = f"id={int(row['order_id'])}_col={row['col']}_row={row['row']}"
        return x, y_filled, mask, info


def filter_meta_by_valid_frac(meta_df, patch_root, target_channel, nodata_value=None, min_valid_frac=0.05):
    keep = []
    for _, r in meta_df.iterrows():
        arr = np.load(patch_root / r["filename"])
        y = arr[target_channel:target_channel+1].astype("float32")
        m = build_target_mask(y, nodata_value=nodata_value)
        keep.append(float(m.mean()) >= min_valid_frac)
    return meta_df.loc[keep].reset_index(drop=True)


# ----------------------------
# STEP 5: Filter (SAME)
# ----------------------------
MIN_VALID_FRAC = 0.05
meta_trainable = filter_meta_by_valid_frac(
    meta_split, PATCH_ROOT,
    target_channel=TARGET_CHANNEL,
    nodata_value=TARGET_NODATA,
    min_valid_frac=MIN_VALID_FRAC
)

print("After filtering:\n", meta_trainable["split"].value_counts())


# ----------------------------
# STEP 6: DEM stats from TRAIN ONLY (SAME AS YOUR CODE)
# IMPORTANT: you used meta_split (not filtered) for DEM stats
# ----------------------------
train_meta_for_dem = meta_split[meta_split["split"] == "train"].reset_index(drop=True)

deltas, all_vals = [], []
for _, r in tqdm(train_meta_for_dem.iterrows(), total=len(train_meta_for_dem), desc="TRAIN DEM stats"):
    dem = np.load(PATCH_ROOT / r["filename"])[DEM_RAW_CHANNEL].astype(np.float32)
    valid = np.isfinite(dem) & (dem != DEM_NODATA)
    if not valid.any():
        continue
    dv = dem[valid]
    all_vals.append(dv)
    deltas.append(float(dv.max() - dv.min()))

all_vals = np.concatenate(all_vals)
deltas = np.array(deltas)

DEM_CENTER_MEDIAN = float(np.median(all_vals))
print("DEM_CENTER_MEDIAN:", DEM_CENTER_MEDIAN)
print("DEM_SCALE_FIXED  :", DEM_SCALE_FIXED)


# ----------------------------
# STEP 7: Compute RGBNIR stats from RAW TRAIN ONLY (SAME)
# ----------------------------
@torch.no_grad()
def compute_input_stats(loader, device="cpu"):
    total_sum = torch.zeros(4, device=device)
    total_sumsq = torch.zeros(4, device=device)
    total_count = 0

    for x, y, mask, info in loader:
        x = x.to(device)
        x = x[:, :4]  # RGBNIR only
        b, c, h, w = x.shape
        total_sum += x.sum(dim=(0,2,3))
        total_sumsq += (x * x).sum(dim=(0,2,3))
        total_count += b * h * w

    mean = total_sum / total_count
    var  = (total_sumsq / total_count) - mean**2
    std  = torch.sqrt(torch.clamp(var, min=1e-12))
    return mean.cpu(), std.cpu()

BATCH_SIZE_STATS = 4

train_ds_raw = NpyPatchDataset(
    meta_trainable, "train", PATCH_ROOT,
    target_nodata=TARGET_NODATA,
    x_mean=None, x_std=None,
    augment=False,
    dem_center=DEM_CENTER_MEDIAN, dem_scale=DEM_SCALE_FIXED  # DEM handling matches, though stats only use RGBNIR
)
train_loader_raw = DataLoader(train_ds_raw, batch_size=BATCH_SIZE_STATS, shuffle=False, num_workers=0)

x_mean, x_std = compute_input_stats(train_loader_raw, device="cpu")
x_mean_t = x_mean.float()
x_std_t  = x_std.float()
print("RAW train RGBNIR mean:", x_mean.numpy())
print("RAW train RGBNIR std :", x_std.numpy())


# ----------------------------
# STEP 8: Build VAL/TEST loaders (NO AUGMENT, SAME)
# ----------------------------
BATCH_SIZE = 8

val_ds = NpyPatchDataset(
    meta_trainable, "val", PATCH_ROOT,
    target_nodata=TARGET_NODATA,
    x_mean=x_mean_t, x_std=x_std_t,
    augment=False,
    dem_center=DEM_CENTER_MEDIAN, dem_scale=DEM_SCALE_FIXED
)
test_ds = NpyPatchDataset(
    meta_trainable, "test", PATCH_ROOT,
    target_nodata=TARGET_NODATA,
    x_mean=x_mean_t, x_std=x_std_t,
    augment=False,
    dem_center=DEM_CENTER_MEDIAN, dem_scale=DEM_SCALE_FIXED
)

val_loader  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


# ----------------------------
# STEP 9: Model definition (SAME ARCH)
# ----------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, p_drop=0.0):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity(),
            nn.Conv2d(mid_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity(),
        )
    def forward(self, x): return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels, p_drop=0.0):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels, p_drop=p_drop))
    def forward(self, x): return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True, p_drop=0.0):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, mid_channels=in_channels // 2, p_drop=p_drop)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels, p_drop=p_drop)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size(2) - x1.size(2)
        diffX = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diffX//2, diffX-diffX//2, diffY//2, diffY-diffY//2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False, p_drop_shallow=0.05, p_drop_deep=0.20):
        super().__init__()
        self.inc = DoubleConv(n_channels, 64, p_drop=p_drop_shallow)
        self.down1 = Down(64, 128, p_drop=p_drop_shallow)
        self.down2 = Down(128, 256, p_drop=p_drop_deep)
        self.down3 = Down(256, 512, p_drop=p_drop_deep)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor, p_drop=p_drop_deep)
        self.up1 = Up(1024, 512 // factor, bilinear, p_drop=p_drop_deep)
        self.up2 = Up(512, 256 // factor, bilinear, p_drop=p_drop_deep)
        self.up3 = Up(256, 128 // factor, bilinear, p_drop=p_drop_shallow)
        self.up4 = Up(128, 64, bilinear, p_drop=p_drop_shallow)
        self.outc = OutConv(64, n_classes)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

IN_CHANNELS = 5
OUT_CHANNELS = 1
model = UNet(n_channels=IN_CHANNELS, n_classes=OUT_CHANNELS, bilinear=False).to(device)
print("Model ready.")


# ----------------------------
# STEP 10: Load BEST checkpoint (SAME LOAD FUNCTION)
# ----------------------------
def load_checkpoint(path, model, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model_state"])
    return ckpt

ckpt = load_checkpoint(BEST_PATH, model, map_location=device)
model.eval()
print("✅ Loaded BEST checkpoint:", BEST_PATH)


# ----------------------------
# STEP 11: Global pixel-wise evaluation (SAME AS YOUR FUNCTION)
# ----------------------------
@torch.no_grad()
def evaluate_global_pixelwise(model, loader, device, eps=1e-8, use_amp=False):
    model.eval()

    # pass 1: global y_mean on valid pixels
    y_sum, y_n = 0.0, 0.0
    for xb, yb, mb, info in loader:
        yb = yb.to(device, non_blocking=True)
        mb = mb.to(device, non_blocking=True)
        valid = mb > 0.5
        yv = yb[valid]
        if yv.numel():
            y_sum += yv.sum().item()
            y_n += yv.numel()
    y_mean = y_sum / max(y_n, 1.0)

    # pass 2: SSE/SAE/SST
    sse, sae, sst, n = 0.0, 0.0, 0.0, 0.0
    for xb, yb, mb, info in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        mb = mb.to(device, non_blocking=True)

        if use_amp and device.type == "cuda":
            with autocast("cuda"):
                pred = model(xb)
        else:
            pred = model(xb)

        valid = mb > 0.5
        yv = yb[valid]
        pv = pred[valid]
        if yv.numel() == 0:
            continue

        diff = pv - yv
        sse += (diff * diff).sum().item()
        sae += diff.abs().sum().item()
        sst += ((yv - y_mean) ** 2).sum().item()
        n += yv.numel()

    mse = sse / max(n, 1.0)
    rmse = mse ** 0.5
    mae = sae / max(n, 1.0)
    r2 = 1.0 - (sse / max(sst, eps))
    return {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2, "n_valid": int(n)}

val_global  = evaluate_global_pixelwise(model, val_loader,  device, use_amp=use_amp)
test_global = evaluate_global_pixelwise(model, test_loader, device, use_amp=use_amp)

print("\n📌 GLOBAL PIXEL-WISE METRICS (BEST checkpoint)")
print(f"VAL  | RMSE: {val_global['rmse']:.4f} dB | MAE: {val_global['mae']:.4f} dB | R²: {val_global['r2']:.4f} | N_valid: {val_global['n_valid']}")
print(f"TEST | RMSE: {test_global['rmse']:.4f} dB | MAE: {test_global['mae']:.4f} dB | R²: {test_global['r2']:.4f} | N_valid: {test_global['n_valid']}")


# ----------------------------
# STEP 12: BW inference dataset (SAME AS YOURS)
# ----------------------------
NODATA = -9999.0

class InferenceNPZDataset(Dataset):
    """
    Loads BW .npz with:
      X = [B02,B03,B04,B08,DEM,NOISE]
      Y = GT noise (H,W) or (1,H,W)
    Builds model input: [R,G,B,NIR,DEM] = [B04,B03,B02,B08,DEM]
    Applies SAME:
      - DEM nodata fill with per-patch median
      - DEM scaling: (DEM - DEM_CENTER_MEDIAN)/DEM_SCALE_FIXED
      - RGBNIR normalization using x_mean_t/x_std_t from TRAIN ONLY
      - target mask with mask_zero_target=True and y!=0.0
    """
    def __init__(self, files, x_mean_t, x_std_t, dem_center, dem_scale,
                 dem_nodata=-9999.0, mask_zero_target=True):
        self.files = files
        self.x_mean = x_mean_t.float()
        self.x_std  = x_std_t.float()
        self.dem_center = float(dem_center)
        self.dem_scale  = float(dem_scale)
        self.dem_nodata = float(dem_nodata)
        self.mask_zero_target = mask_zero_target

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        p = self.files[idx]
        d = np.load(p)
        X = d["X"].astype(np.float32)
        Y = d["Y"].astype(np.float32)
        if Y.ndim == 2:
            Y = Y[None, :, :]

        b02, b03, b04, b08, dem = X[0], X[1], X[2], X[3], X[4]
        x = np.stack([b04, b03, b02, b08, dem], axis=0).astype(np.float32)

        # DEM fill (median)
        dem_valid = np.isfinite(x[4]) & (x[4] != self.dem_nodata)
        dem_fill = np.median(x[4][dem_valid]) if dem_valid.any() else 0.0
        x[4][~dem_valid] = dem_fill

        # DEM scaling
        x[4] = (x[4] - self.dem_center) / (self.dem_scale + 1e-6)

        # RGBNIR normalization
        mean = self.x_mean.view(4,1,1).numpy()
        std  = self.x_std.view(4,1,1).numpy()
        x[:4] = (x[:4] - mean) / (std + 1e-6)

        # target + mask
        y = Y.astype(np.float32)
        mask = np.isfinite(y) & (y > -1e6)
        if self.mask_zero_target:
            mask = mask & (y != 0.0)
        mask = mask & (y != NODATA)
        mask = mask.astype(np.float32)

        y_filled = y.copy()
        y_filled[mask == 0] = 0.0

        return torch.from_numpy(x), torch.from_numpy(y_filled), torch.from_numpy(mask), p.name

inf_ds = InferenceNPZDataset(
    INF_FILES,
    x_mean_t=x_mean_t, x_std_t=x_std_t,
    dem_center=DEM_CENTER_MEDIAN,
    dem_scale=DEM_SCALE_FIXED,
    dem_nodata=-9999.0,
    mask_zero_target=True
)
inf_loader = DataLoader(inf_ds, batch_size=1, shuffle=False, num_workers=0)

inf_global = evaluate_global_pixelwise(model, inf_loader, device, use_amp=use_amp)
print("\n📌 BW INFERENCE METRICS (masked valid pixels only)")
print(f"BW   | RMSE: {inf_global['rmse']:.4f} dB | MAE: {inf_global['mae']:.4f} dB | R²: {inf_global['r2']:.4f} | N_valid: {inf_global['n_valid']}")

In [ ]:
import numpy as np
import torch

@torch.no_grad()
def per_patch_metrics(model, ds, device, use_amp=False):
    model.eval()
    out = []
    for i in range(len(ds)):
        x, y, mask, name = ds[i]
        xb = x.unsqueeze(0).to(device)
        yb = y.unsqueeze(0).to(device)
        mb = mask.unsqueeze(0).to(device)

        if use_amp and device.type == "cuda":
            from torch.amp import autocast
            with autocast("cuda"):
                pred = model(xb)
        else:
            pred = model(xb)

        valid = mb > 0.5
        yv = yb[valid]
        pv = pred[valid]
        n = int(yv.numel())

        if n == 0:
            out.append({"patch": name, "rmse": np.nan, "mae": np.nan, "r2": np.nan, "n_valid": 0})
            continue

        diff = pv - yv
        sse = float((diff * diff).sum().item())
        sae = float(diff.abs().sum().item())

        rmse = (sse / n) ** 0.5
        mae = sae / n

        y_mean = float(yv.mean().item())
        sst = float(((yv - y_mean) ** 2).sum().item())
        r2 = 1.0 - (sse / sst) if sst > 1e-12 else np.nan

        out.append({"patch": name, "rmse": rmse, "mae": mae, "r2": r2, "n_valid": n})

    return out

bw_patch_stats = per_patch_metrics(model, inf_ds, device=device, use_amp=use_amp)
for r in bw_patch_stats:
    print(f"{r['patch']:20s} | RMSE {r['rmse']:.3f} | MAE {r['mae']:.3f} | R2 {r['r2']:.3f} | N {r['n_valid']}")

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def scatter_bw_gt_vs_pred(model, ds, device, use_amp=False, max_points=120_000):
    model.eval()
    gts, prs = [], []

    for i in range(len(ds)):
        x, y, mask, name = ds[i]
        xb = x.unsqueeze(0).to(device)
        yb = y.unsqueeze(0).to(device)
        mb = mask.unsqueeze(0).to(device)

        if use_amp and device.type == "cuda":
            from torch.amp import autocast
            with autocast("cuda"):
                pred = model(xb)
        else:
            pred = model(xb)

        valid = mb > 0.5
        gts.append(yb[valid].detach().cpu().numpy())
        prs.append(pred[valid].detach().cpu().numpy())

    gt = np.concatenate(gts) if gts else np.array([])
    pr = np.concatenate(prs) if prs else np.array([])

    if gt.size == 0:
        print("No valid pixels for scatter.")
        return

    # downsample for readability
    if gt.size > max_points:
        idx = np.random.default_rng(42).choice(gt.size, size=max_points, replace=False)
        gt, pr = gt[idx], pr[idx]

    plt.figure(figsize=(6,6))
    plt.scatter(gt, pr, s=2, alpha=0.2)
    mn = float(min(gt.min(), pr.min()))
    mx = float(max(gt.max(), pr.max()))
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("GT noise (dB)")
    plt.ylabel("Pred noise (dB)")
    plt.title("BW generalization: GT vs Pred (valid pixels)")
    plt.tight_layout()
    plt.show()

scatter_bw_gt_vs_pred(model, inf_ds, device=device, use_amp=use_amp)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# ----------------------------
# 1) Per-patch metrics (if not computed already)
# ----------------------------
@torch.no_grad()
def per_patch_metrics(model, ds, device, use_amp=False):
    model.eval()
    out = []
    for i in range(len(ds)):
        x, y, mask, name = ds[i]
        xb = x.unsqueeze(0).to(device)
        yb = y.unsqueeze(0).to(device)
        mb = mask.unsqueeze(0).to(device)

        if use_amp and device.type == "cuda":
            from torch.amp import autocast
            with autocast("cuda"):
                pred = model(xb)
        else:
            pred = model(xb)

        valid = mb > 0.5
        yv = yb[valid]
        pv = pred[valid]
        n = int(yv.numel())

        if n == 0:
            out.append({"patch": name, "rmse": np.nan, "mae": np.nan, "r2": np.nan, "n_valid": 0})
            continue

        diff = pv - yv
        sse = float((diff * diff).sum().item())
        sae = float(diff.abs().sum().item())

        rmse = (sse / n) ** 0.5
        mae = sae / n

        y_mean = float(yv.mean().item())
        sst = float(((yv - y_mean) ** 2).sum().item())
        r2 = 1.0 - (sse / sst) if sst > 1e-12 else np.nan

        out.append({"patch": name, "rmse": rmse, "mae": mae, "r2": r2, "n_valid": n})

    return out


import numpy as np
import matplotlib.pyplot as plt
import torch

def denorm_rgb_from_input(x_tensor, x_mean_t, x_std_t):
    x = x_tensor.clone()
    mean = x_mean_t.view(4,1,1).to(x.device)
    std  = x_std_t.view(4,1,1).to(x.device)
    x[:4] = x[:4] * std + mean
    rgb = x[:3].permute(1,2,0).detach().cpu().numpy()
    lo = np.percentile(rgb, 2, axis=(0,1), keepdims=True)
    hi = np.percentile(rgb, 98, axis=(0,1), keepdims=True)
    rgb = (rgb - lo) / (hi - lo + 1e-6)
    return np.clip(rgb, 0, 1)

@torch.no_grad()
def visualize_bw_metrics_panel_5col(
    model, ds, indices, device,
    x_mean_t, x_std_t,
    dem_center, dem_scale,
    patch_stats=None,
    suptitle="Baden-Württemberg generalization (Metrics | RGB | DEM | GT | Pred)",
    fixed_noise_range=None,
    metrics_col_width=1.15
):
    """
    fixed_noise_range: None or (vmin, vmax)
    metrics_col_width: widen/narrow the metrics column relative to others
    """
    model.eval()

    # lookup stats
    stat_by_name = {}
    if patch_stats is not None:
        stat_by_name = {s["patch"]: s for s in patch_stats}

    n = len(indices)

    # 5 columns with custom width ratios (make metrics a bit wider)
    width_ratios = [metrics_col_width, 1.2, 1.0, 1.0, 1.0]
    fig, axes = plt.subplots(
        n, 5, figsize=(18, 3*n),
        gridspec_kw={"width_ratios": width_ratios},
        constrained_layout=True
    )
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_i, idx in enumerate(indices):
        x, y, mask, name = ds[idx]
        xb = x.unsqueeze(0).to(device)

        pred = model(xb).squeeze(0).squeeze(0).detach().cpu().numpy()
        gt = y.squeeze(0).detach().cpu().numpy()
        m  = (mask.squeeze(0).detach().cpu().numpy() > 0.5)
        invalid = ~m

        # ----- RGB -----
        rgb = denorm_rgb_from_input(x, x_mean_t, x_std_t)

        # ----- DEM (meters) -----
        dem_scaled = x[4].detach().cpu().numpy()
        dem_m = dem_scaled * (dem_scale + 1e-6) + dem_center
        dmin = np.percentile(dem_m, 2)
        dmax = np.percentile(dem_m, 98)

        # ----- GT valid only -----
        gt_disp = gt.copy()
        gt_disp[invalid] = np.nan

        # ----- noise scale -----
        if fixed_noise_range is not None:
            vmin, vmax = fixed_noise_range
        else:
            valid_vals = gt[m]
            if valid_vals.size > 0:
                vmin = np.percentile(valid_vals, 2)
                vmax = np.percentile(valid_vals, 98)
            else:
                vmin, vmax = None, None

        # =========================
        # COL 0: METRICS PANEL
        # =========================
        axM = axes[row_i, 0]
        axM.axis("off")

        if name in stat_by_name:
            s = stat_by_name[name]
            metric_text = (
                f"{name}\n\n"
                f"RMSE : {s['rmse']:.2f} dB\n"
                f"MAE  : {s['mae']:.2f} dB\n"
                f"R²   : {s['r2']:.2f}\n"
                f"N    : {s['n_valid']}\n"
            )
        else:
            metric_text = f"{name}\n\nMetrics: N/A"

        axM.text(
            0.02, 0.98, metric_text,
            transform=axM.transAxes,
            va="top", ha="left",
            fontsize=16, weight="bold",
            bbox=dict(facecolor="white", edgecolor="black", linewidth=1.2, pad=8.0)
        )

        # =========================
        # COL 1: RGB
        # =========================
        ax0 = axes[row_i, 1]
        ax0.imshow(rgb)
        ax0.set_title("RGB")
        ax0.axis("off")

        # =========================
        # COL 2: DEM
        # =========================
        ax1 = axes[row_i, 2]
        im_dem = ax1.imshow(dem_m, cmap="terrain", vmin=dmin, vmax=dmax)
        ax1.set_title("DEM (m)")
        ax1.axis("off")
        plt.colorbar(im_dem, ax=ax1, fraction=0.046, pad=0.02)

        # =========================
        # COL 3: GT
        # =========================
        ax2 = axes[row_i, 3]
        im_gt = ax2.imshow(gt_disp, cmap="viridis", vmin=vmin, vmax=vmax)
        ax2.set_title("GT Noise (valid only)")
        ax2.axis("off")
        plt.colorbar(im_gt, ax=ax2, fraction=0.046, pad=0.02)

        # =========================
        # COL 4: Pred
        # =========================
        ax3 = axes[row_i, 4]
        im_pr = ax3.imshow(pred, cmap="viridis", vmin=vmin, vmax=vmax)
        ax3.set_title("Pred Noise")
        ax3.axis("off")
        plt.colorbar(im_pr, ax=ax3, fraction=0.046, pad=0.02)

    fig.suptitle(suptitle, fontsize=14)
    plt.show()
# 4) Run it for your 4 BW patches
# ----------------------------
# compute stats once (if not already)
bw_patch_stats = per_patch_metrics(model, inf_ds, device=device, use_amp=use_amp)
vis_idx = list(range(len(inf_ds)))

visualize_bw_metrics_panel_5col(
    model, inf_ds, vis_idx, device,
    x_mean_t, x_std_t,
    DEM_CENTER_MEDIAN, DEM_SCALE_FIXED,
    patch_stats=bw_patch_stats,
    fixed_noise_range=None,       # or (60, 75) for fixed scale
    metrics_col_width=1.25        # increase if you want more space
)
# Option B: fixed 60–75 dB (more “fair” across patches)
# visualize_bw_4col_with_box(
#     model, inf_ds, vis_idx, device,
#     x_mean_t, x_std_t,
#     DEM_CENTER_MEDIAN, DEM_SCALE_FIXED,
#     patch_stats=bw_patch_stats,
#     fixed_noise_range=(60, 75)
# )